In [3]:
import torch
import torch.nn as nn

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, heads, dims, attn_scale=None, sr_scale=1):
        super().__init__()
        # 确认维度是否正好能平分给每个注意力头
        assert dims % heads == 0, f"dim {dims} should be divided by num_heads {heads}."
        
        # 确认每个注意力头处理的特征维度
        self.dims = dims
        self.heads = heads
        self.dim_per_head = dims // heads
        
        # 定义获得query、key、value的投影张量
        self.proj_q = nn.Linear(dims, dims)
        self.proj_k = nn.Linear(dims, dims)
        self.proj_v = nn.Linear(dims, dims)
        
        # 考虑下采样
        self.sr = None
        if sr_scale > 1:
            self.sr = nn.Conv2d(dims, dims, kernel_size=sr_scale, stride=sr_scale)
            self.norm = nn.LayerNorm(dims)
        
        # 定义投影层
        self.proj = nn.Linear(dims, dims)
        
    def forward(self, x):
        # 得到输入的尺寸
        batch_size, height, width, dim = x.shape
        x = x.reshape(batch_size, -1, dim)
        
        # 获取Q、K、V
        self.query = self.proj_q(x).reshape(batch_size, dim, self.heads, self.dim_per_head)
        

        # Q、K相乘，经softmax函数得到注意力分数

        # value加权，获得最终输出